# ecDNA velocity analysis

This notebook provides a minimal workflow for ecDNA velocity analysis from two cell-by-ecDNA matrices. 

Input matrices:

- **Supporting matrix**: read counts supporting ecDNA breakpoints or junctions in each cell.
- **Chromosome matrix**: read counts from matched chromosomal background regions that do not support ecDNA junctions.
- Rows are cell barcodes and columns are ecDNA breakpoints.



## 1. Load packages

In [ ]:
# Load required packages.
suppressPackageStartupMessages({
  library(Matrix)
  library(tibble)
  library(dplyr)
  library(pagoda2)
  library(velocyto.R)
  library(igraph)
})

# Source the local plotting helper if you use the customized show.velocity() script.
show_velocity_file <- "show.velocity.r"
if (file.exists(path.expand(show_velocity_file))) {
  source(path.expand(show_velocity_file))
}

## 2. Set default parameters


In [ ]:
# Analysis label.
gene <- "EGFR"

# Current single-run defaults used in this tutorial.
distance_method <- "cosine"
knn_k <- 20
fit_quantile <- 0.01
k_cells <- 20
n_arrows <- 200

# Input files: replace these paths with your exported matrices.
supporting_file <- "supportingmatrix"
chromosome_file <- "chromosomematrix"

# Output folder.
outdir <- file.path("velocity_analysis")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)


## 3. Read and clean matrices

In [ ]:
# Stop early if the two input files are not available.
stopifnot(file.exists(supporting_file))
stopifnot(file.exists(chromosome_file))

# Read cell-by-feature matrices.
supporting_reads <- read.csv(supporting_file, check.names = FALSE)
chromosome_reads <- read.csv(chromosome_file, check.names = FALSE)

# Use the first column as barcode when Barcode is not explicitly present.
if (!"Barcode" %in% colnames(supporting_reads)) {
  colnames(supporting_reads)[1] <- "Barcode"
}
if (!"Barcode" %in% colnames(chromosome_reads)) {
  colnames(chromosome_reads)[1] <- "Barcode"
}

# Remove the known problematic barcode and keep shared cells only.
shared_cells <- intersect(supporting_reads$Barcode, chromosome_reads$Barcode)

supporting_reads <- supporting_reads %>% filter(Barcode %in% shared_cells)
chromosome_reads <- chromosome_reads %>% filter(Barcode %in% shared_cells)

# Convert data frames to numeric matrices with cells as rows.
support_mat <- supporting_reads %>%
  column_to_rownames("Barcode") %>%
  mutate(across(everything(), as.numeric)) %>%
  as.matrix()

chromosome_mat <- chromosome_reads %>%
  column_to_rownames("Barcode") %>%
  mutate(across(everything(), as.numeric)) %>%
  as.matrix()

# Convert to feature-by-cell sparse matrices for velocyto-style analysis.
emat <- t(as(support_mat, "dgCMatrix"))
nmat <- t(as(chromosome_mat, "dgCMatrix"))

# Keep cells with at least one supporting read.
emat <- emat[, Matrix::colSums(emat) >= 1]
nmat <- nmat[, colnames(emat)]

cat("supporting matrix:", dim(emat), "\n")
cat("chromosome matrix:", dim(nmat), "\n")

## 4. Build cell graph with pagoda2

In [ ]:
# Build a pagoda2 object using the supporting-read matrix.
r <- Pagoda2$new(emat, n.cores = 1, trim = FALSE, min.transcripts.per.cell = 1)
r$adjustVariance(plot = FALSE, do.par = TRUE, gam.k = 1)

# Calculate PCA, KNN graph, clusters, and tSNE embedding.
r$calculatePcaReduction(nPcs = 20, n.odgenes = 3000, maxit = 100)
r$makeKnnGraph(k = knn_k, type = "PCA", center = TRUE, distance = distance_method)
r$getKnnClusters(method = multilevel.community, type = "PCA", name = "multilevel", resolution = 0.01)
r$getEmbedding(type = "PCA", embeddingType = "tSNE", perplexity = 50, verbose = TRUE)

# Restrict matrices to cells retained by pagoda2.
emat <- emat[, rownames(r$counts)]
nmat <- nmat[, rownames(r$counts)]

# Use the pagoda2 tSNE embedding for plotting.
emb <- r$embeddings$PCA$tSNE

cat("cells after pagoda2 filtering:", ncol(emat), "\n")

## 5. Calculate EGFR ecDNA ratio for cell colors

In [ ]:
# Estimate ecDNA ratio from supporting and chromosome reads.
support_sum <- Matrix::colSums(emat)
chromosome_sum <- Matrix::colSums(nmat)
ecDNA_ratio <- support_sum / pmax(support_sum + chromosome_sum, 1)
ecDNA_ratio <- pmin(pmax(ecDNA_ratio, 0), 1)

# Map ecDNA ratio to a warm color scale.
heatmap_colors <- colorRampPalette(c("#FFF5E6", "#FFE0B2", "#FFAB78", "#FF7043", "#E64A19", "#BF360C"))(100)
color_index <- round(ecDNA_ratio * 99) + 1
cell_colors <- heatmap_colors[color_index]
names(cell_colors) <- names(ecDNA_ratio)

summary(ecDNA_ratio)

## 6. Estimate ecDNA velocity

In [ ]:
# Calculate cell-cell distance from PCA coordinates.
cell_dist <- as.dist(1 - armaCor(t(r$reductions$PCA)))

# Estimate relative velocity using supporting and chromosome matrices.
rvel_cd <- gene.relative.velocity.estimates(
  emat,
  nmat,
  deltaT = 1,
  kCells = k_cells,
  min.nmat.emat.correlation = 0.2,
  min.nmat.emat.slope = 0.2,
  cell.dist = cell_dist,
  fit.quantile = fit_quantile,
  n.cores = 1
)

## 7. Plot and save velocity map

In [ ]:
# Save the velocity plot as a PDF.
pdf_file <- file.path(
  outdir,
  paste0("ecDNA_velocity_", distance_method, "_KnnGraph_k", knn_k,
         "_fit", fit_quantile, "_kcells", k_cells, "_n", n_arrows, ".pdf")
)

pdf(pdf_file, width = 2.5, height = 2.5, useDingbat = FALSE)
show.velocity(
  emb,
  rvel_cd,
  n = n_arrows,
  scale = "sqrt",
  cell.colors = ac(x = cell_colors[rownames(emb)], alpha = 0.4),
  cex = 0.5,
  arrow.scale = 5,
  grid.arrow.color = "#1A1A2E",
  show.grid.flow = TRUE,
  min.grid.cell.mass = 0.4,
  grid.n = 25,
  arrow.lwd = 0.4,
  do.par = TRUE,
  cell.border.alpha = 0
)
dev.off()

cat("Saved velocity plot to:", pdf_file, "\n")